# Análise de Dados

Responde às quatro perguntas de negócio formuladas em `datasets.md`.

**Nota metodológica que atravessa toda a análise:** as métricas de investimento são sempre
usadas *per capita*. Em valores absolutos, a correlação do Bolsa Família com o IDEB é de
-0,041 - ruído, porque o valor absoluto apenas reproduz o tamanho da população. Per capita,
sobe para -0,490.

In [0]:
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

# Parâmetro: usado pelo Job (Jobs & Pipelines) e com padrão para execução interativa.
dbutils.widgets.text("catalog", "workspace")

CATALOG = dbutils.widgets.get("catalog")

spark.sql(f"USE CATALOG {CATALOG}")

## Tabela analítica

Junta os três fatos pela dimensão de município. O IDEB usa a etapa Anos Finais, que é a
mais sensível a fatores socioeconômicos.

In [0]:
df_analise = (
    spark.table(f"{CATALOG}.gold.dim_municipio").alias("m")
    .join(
        spark.table(f"{CATALOG}.gold.fato_desempenho_educacional")
             .where(F.col("etapa_ensino") == "Anos Finais")
             .select("codigo_municipio_ibge", "vl_ideb", "vl_nota_matematica", "vl_nota_portugues"),
        "codigo_municipio_ibge", "left",
    )
    .join(
        spark.table(f"{CATALOG}.gold.fato_infraestrutura_escolar").drop("ano"),
        "codigo_municipio_ibge", "left",
    )
    .join(
        spark.table(f"{CATALOG}.gold.fato_investimento_social")
             .select("codigo_municipio_ibge", "valor_per_capita", "taxa_cobertura_pct"),
        "codigo_municipio_ibge", "left",
    )
    .where(F.col("vl_ideb").isNotNull())
)

# serverless não suporta cache/persist; a tabela é pequena e o recálculo é barato
print(f"{df_analise.count():,} municípios com IDEB e dados completos")
pdf = df_analise.toPandas()

## Pergunta 1 - Municípios com maior Bolsa Família per capita têm IDEB maior ou menor?

In [0]:
media_nacional = pdf["vl_ideb"].mean()
r_bf = pdf["vl_ideb"].corr(pdf["valor_per_capita"])
print(f"IDEB médio nacional (Anos Finais, rede pública): {media_nacional:.2f}")
print(f"Correlação IDEB x Bolsa Família per capita:      {r_bf:+.3f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(pdf["valor_per_capita"], pdf["vl_ideb"], s=6, alpha=0.25)
ax.set_xlabel("Bolsa Família per capita (R$/hab no mês de referência)")
ax.set_ylabel("IDEB - Anos Finais")
ax.set_title("Investimento social x desempenho educacional - municípios brasileiros")
ax.axhline(media_nacional, linestyle="--", linewidth=1, color="grey")
display(fig)

### Gradiente por quartil

O coeficiente sozinho esconde a forma da relação; o quartil mostra.

In [0]:
import pandas as pd

pdf["quartil_bf"] = pd.qcut(pdf["valor_per_capita"], 4,
                            labels=["Q1 (menor)", "Q2", "Q3", "Q4 (maior)"])
display(
    pdf.groupby("quartil_bf", observed=True)
       .agg(municipios=("vl_ideb", "size"),
            ideb_medio=("vl_ideb", "mean"),
            bf_per_capita=("valor_per_capita", "mean"),
            populacao_media=("populacao", "mean"))
       .round(2)
)

## Pergunta 2 - Escolas com melhor infraestrutura têm melhor desempenho?

> A pergunta foi originalmente formulada em `datasets.md` sobre os repasses do PDDE (FNDE).
> Essa fonte não entrou no pipeline; a pergunta é respondida pela via da infraestrutura
> física efetivamente observada nos microdados do Censo Escolar. A ausência do PDDE é
> registrada na Autoavaliação - a pergunta não é removida, conforme a regra do edital.

In [0]:
INDICADORES = [
    "pct_escolas_internet", "pct_escolas_laboratorio_informatica", "pct_escolas_biblioteca",
    "pct_escolas_laboratorio_ciencias", "pct_escolas_quadra_esportes", "pct_escolas_refeitorio",
    "pct_escolas_esgoto_rede_publica", "pct_escolas_agua_potavel", "pct_escolas_acessibilidade_rampas",
    "media_alunos_por_docente",
]

correlacoes = sorted(
    ((ind, pdf["vl_ideb"].corr(pdf[ind])) for ind in INDICADORES),
    key=lambda x: -abs(x[1]),
)
for indicador, r in correlacoes:
    print(f"{indicador:38} r = {r:+.3f}")

fig, ax = plt.subplots(figsize=(9, 5))
nomes = [i.replace("pct_escolas_", "").replace("_", " ") for i, _ in correlacoes]
ax.barh(nomes[::-1], [r for _, r in correlacoes][::-1])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Correlação com o IDEB")
ax.set_title("Infraestrutura escolar x desempenho")
display(fig)

## Pergunta 3 - Existe correlação entre investimento e desempenho?

In [0]:
print("Matriz de correlação com o IDEB:\n")
for coluna in ["valor_per_capita", "taxa_cobertura_pct", "pct_escolas_internet",
               "pct_escolas_biblioteca", "qt_escolas_publicas", "populacao"]:
    print(f"  {coluna:26} r = {pdf['vl_ideb'].corr(pdf[coluna]):+.3f}")

## Pergunta 4 - Como essas relações variam entre regiões e UFs?

Calcular a correlação *dentro* de cada região revela se a relação nacional se sustenta
localmente ou se é efeito de composição entre regiões.

In [0]:
por_regiao = pdf.groupby("nome_regiao").agg(
    municipios=("vl_ideb", "size"),
    ideb_medio=("vl_ideb", "mean"),
    bf_per_capita=("valor_per_capita", "mean"),
    pct_internet=("pct_escolas_internet", "mean"),
)
# correlação por grupo calculada à parte: evita groupby.apply, cuja assinatura mudou no pandas 2.2
por_regiao["r_bf_ideb"] = pdf.groupby("nome_regiao").apply(
    lambda d: d["vl_ideb"].corr(d["valor_per_capita"])
)
por_regiao = por_regiao.round(2).sort_values("ideb_medio", ascending=False)
display(por_regiao)

fig, ax = plt.subplots(figsize=(9, 5))
for regiao, dados in pdf.groupby("nome_regiao"):
    ax.scatter(dados["valor_per_capita"], dados["vl_ideb"], s=6, alpha=0.35, label=regiao)
ax.set_xlabel("Bolsa Família per capita (R$/hab)")
ax.set_ylabel("IDEB - Anos Finais")
ax.set_title("A mesma relação, por região")
ax.legend(markerscale=3)
display(fig)

In [0]:
display(
    df_analise.groupBy("sigla_uf")
    .agg(
        F.count("*").alias("municipios"),
        F.round(F.avg("vl_ideb"), 2).alias("ideb_medio"),
        F.round(F.avg("valor_per_capita"), 2).alias("bf_per_capita"),
        F.round(F.avg("pct_escolas_internet"), 1).alias("pct_internet"),
    )
    .orderBy(F.desc("ideb_medio"))
)